# BESS optimal placement and sizing

Run this after candidate site selection and prepare a completed hourly system profile.

In [1]:
import json
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "src" / "paths.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate project root containing src/paths.py")
sys.path.insert(0, str(ROOT))
from src.paths import CONFIG, OPTIMIZATION, PROCESSED
from src.bess_optimization import optimize_bess

## Load inputs and solve

The optimizer requires ranked eligible sites and an hourly profile with `timestamp`, `demand_mw`, and `solar_mw`.

In [ ]:
profile = pd.read_csv(
    PROCESSED / "optimization_profile.csv"
)

profile["timestamp"] = pd.to_datetime(
    profile["timestamp"],
    utc=True
)

candidates = pd.read_csv(
    PROCESSED / "candidate_bess_sites.csv"
)

config = json.load(
    open(CONFIG / "model_config.json")
)

print("Profile records:", len(profile))
print("Candidate substations:", len(candidates))

### Run optimization

In [ ]:
results = optimize_bess(
    profile=profile,
    candidates=candidates,
    config=config
)

print("Optimization completed.")

print(
    "Objective value:",
    results["objective_value"]
)

print(
    "Peak residual demand:",
    results["peak_residual_mw"],
    "MW"
)

### Display selected locations

In [ ]:
selected_sites = results["selected_sites"]

display(selected_sites)